In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
def optimize_random_forest_with_PCA_toggle(X, y, pca_toggle=False, n_iter=10, n_PCs=45, random_state=4, printing=True):
    '''
    Optimizes a random forest classifier, optionally applying PCA, and provides feature importance.

    Inputs:
        X: pandas.DataFrame
            Feature matrix for training and testing.
        y: pandas.Series or array-like
            Target labels for classification.
        pca_toggle: bool
            Whether to include PCA dimensionality reduction in the pipeline.

    Returns:
        random_search: RandomizedSearchCV object
            Fitted RandomizedSearchCV instance containing the best model and results.
        X_test, y_test: arrays
            Test data and labels for evaluation.
    '''
    
    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    # Build the pipeline: optional PCA step, then Random Forest
    steps = []
    if pca_toggle:
        steps.append(('pca', PCA()))
    steps.append(('rf', RandomForestClassifier(random_state=42)))
    pipeline = Pipeline(steps)

    # Define the hyperparameter search space for Random Forest
    hyperparameter_space = {
        'rf__n_estimators': randint(100, 500),
        'rf__max_depth': [None, 10, 20, 30, 40, 50],
        'rf__min_samples_split': randint(2, 10),
        'rf__min_samples_leaf': randint(1, 5)
    }

    # If PCA is used, keep the number of components fixed
    if pca_toggle:
        hyperparameter_space['pca__n_components'] = [n_PCs]

    # Run randomized hyperparameter search with 4-fold CV
    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=hyperparameter_space,
        n_iter=n_iter,
        cv=4,
        n_jobs=-1,
        verbose=1,
        random_state=2,
        scoring='f1_macro'  # macro F1 is suitable for multi-class problems
    )

    # Fit the search on the training data
    random_search.fit(X_train, y_train)

    if printing:
        print(f"Best Parameters: {random_search.best_params_}")
        print(f"Best CV Score: {random_search.best_score_:.4f}")

    # Use the best model found during the search
    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)

    if printing:
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

    # Extract feature importances only when PCA is not used
    if not pca_toggle:
        rf_model = best_model.named_steps['rf']
        feature_importances = pd.DataFrame({
            'Feature': X.columns,
            'Importance': rf_model.feature_importances_
        }).sort_values(by='Importance', ascending=False)

        if printing:
            print("\nTop 10 Most Important Features:")
            print(feature_importances.head(10))
    else:
        if printing:
            print("\nPCA enabled - feature importances are not available.")
        feature_importances = None

    return random_search, X_test, y_test, feature_importances



In [ ]:
def repeat_rf_optimization(
    X, y, 
    n_runs=50, 
    pca_toggle=False, 
    n_iter=10, 
    n_PCs=25
):
    """
    Repeatedly fit and evaluate an optimized Random Forest classifier across
    multiple random train/test splits, then aggregate performance statistics.

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix. Rows should correspond to samples and columns to features.

    y : pandas.Series or array-like
        Target labels aligned to `X`.

    n_runs : int, default=50
        Number of repeated optimization/evaluation runs to perform.

    pca_toggle : bool, default=False
        Whether PCA is included in the optimization pipeline.
        If True, feature importances are not aggregated because PCA-transformed
        components are not directly interpretable as original features.

    n_iter : int, default=10
        Number of parameter settings sampled in each RandomizedSearchCV run.

    n_PCs : int, default=25
        Number of principal components to retain if PCA is enabled.

    Returns
    -------
    confusions : list of np.ndarray
        Confusion matrix from each run.

    metrics_df : pandas.DataFrame
        Per-run classification metrics.

    avg_confusion : np.ndarray
        Element-wise mean confusion matrix across all runs.

    avg_importances : pandas.DataFrame or None
        Mean and standard deviation of feature importances across runs when
        PCA is disabled; otherwise None.

    metrics_summary : pandas.DataFrame
        Summary table containing the mean and standard deviation of each metric
        across runs.

    y_true_runs : list
        Ground-truth labels for the test set from each run.

    y_pred_runs : list
        Predicted labels for the test set from each run.

    mouse_id_runs : list
        Sample IDs (taken from the test-set index) for each run.
    """

    # Store run-level outputs so they can be summarized after all repetitions.
    confusions = []
    metrics_list = []
    importances_list = []

    # Store per-run prediction details for downstream inspection, such as
    # identifying which samples were repeatedly misclassified.
    y_true_runs = []
    y_pred_runs = []
    mouse_id_runs = []

    # Fix the label order once so all confusion matrices have consistent shape
    # and class ordering across runs.
    labels = np.unique(y)

    # ------------------------------------------------------------------
    # Repeat the full optimization/evaluation workflow multiple times
    # using different random seeds to assess model robustness.
    # ------------------------------------------------------------------
    for i in range(n_runs):
        print(f"Run {i+1}/{n_runs}\n")

        # Run the RF optimization pipeline for this iteration.
        # The helper function is expected to return:
        # - rs: fitted RandomizedSearchCV object
        # - X_test: held-out test features
        # - y_test: held-out test labels
        # - feature_importances: importance table for the best RF model
        rs, X_test, y_test, feature_importances = optimize_random_forest_with_PCA_toggle(
            X, y, pca_toggle=pca_toggle, n_iter=n_iter, n_PCs=n_PCs, random_state=i + 42
        )

        # Extract the best model found during hyperparameter search and use it
        # to generate predictions on the held-out test split.
        best_model = rs.best_estimator_
        y_pred = best_model.predict(X_test)

        # Save the true and predicted labels from this run so results can be
        # inspected later at the sample level.
        y_true_runs.append(y_test.copy())
        y_pred_runs.append(pd.Series(y_pred, index=X_test.index, name="y_pred"))

        # Save sample IDs from the test set index. This is useful for tracing
        # performance back to specific mice/samples across repeated splits.
        mouse_id_runs.append(
            pd.Series(
                X_test.index.astype(str),
                index=X_test.index,
                name=X_test.index.name if X_test.index.name is not None else "mouse_id"
            )
        )

        # Compute and store the confusion matrix for this run using a fixed
        # label order so matrices are directly comparable and averageable.
        cm = confusion_matrix(y_test, y_pred, labels=labels)
        confusions.append(cm)

        # Compute standard classification metrics for this run.
        # Macro averages treat all classes equally.
        # Weighted averages account for class support.
        # Micro averages compute global counts across all classes.
        run_metrics = {
            "run": i + 1,
            "accuracy": accuracy_score(y_test, y_pred),

            "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),

            "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),

            "precision_micro": precision_score(y_test, y_pred, average="micro", zero_division=0),
            "recall_micro": recall_score(y_test, y_pred, average="micro", zero_division=0),
            "f1_micro": f1_score(y_test, y_pred, average="micro", zero_division=0),
        }
        metrics_list.append(run_metrics)

        # If PCA is disabled, feature importances remain interpretable in the
        # original feature space. Reindex to X.columns so all runs align to the
        # same feature order before later aggregation.
        if not pca_toggle:
            fi_series = (
                feature_importances
                .set_index("Feature")["Importance"]
                .reindex(X.columns, fill_value=0.0)
            )
            importances_list.append(fi_series)

    # ------------------------------------------------------------------
    # Aggregate results across all runs
    # ------------------------------------------------------------------

    # Average the confusion matrices element-wise to summarize typical
    # classification behavior across repeated train/test splits.
    avg_confusion = np.mean(confusions, axis=0)

    # Convert the list of per-run metric dictionaries into a DataFrame for
    # easier summary statistics and downstream plotting.
    metrics_df = pd.DataFrame(metrics_list)

    # Build a compact summary table with the mean and standard deviation of
    # each metric across runs.
    metric_cols = [c for c in metrics_df.columns if c != "run"]
    metrics_summary = pd.DataFrame({
        "metric": metric_cols,
        "mean": [metrics_df[c].mean() for c in metric_cols],
        "std":  [metrics_df[c].std(ddof=0) for c in metric_cols],
    }).sort_values("mean", ascending=False).reset_index(drop=True)

    # Print aggregate confusion matrix.
    print("\n=======================================")
    print("Average Confusion Matrix Across Runs")
    print("=======================================")
    print(avg_confusion)

    # Print performance summary in a readable mean ± std format.
    print("\n=======================================")
    print("Performance Summary (Mean ± Std)")
    print("=======================================")
    for c in metric_cols:
        print(f"{c:20s}: {metrics_df[c].mean():.4f} ± {metrics_df[c].std(ddof=0):.4f}")

    # ------------------------------------------------------------------
    # Aggregate feature importances across runs when PCA is not used
    # ------------------------------------------------------------------
    if not pca_toggle:
        # Stack all per-run importance vectors into one DataFrame where each row
        # is a run and each column is a feature.
        importances_df = pd.concat(importances_list, axis=1).T

        # Compute mean and standard deviation of feature importance across runs
        # to identify features that are both important and stable.
        avg_importances = pd.DataFrame({
            "Feature": importances_df.columns,
            "AvgImportance": importances_df.mean(axis=0).values,
            "StdImportance": importances_df.std(axis=0, ddof=0).values
        }).sort_values(by="AvgImportance", ascending=False).reset_index(drop=True)

        print("\n=======================================")
        print("Average Feature Importance Across Runs")
        print("=======================================")
        print(avg_importances.head(20))
    else:
        # Feature importances are not directly meaningful in the original
        # feature space after PCA transformation, so skip aggregation.
        avg_importances = None
        print("\nPCA enabled — skipping feature importance aggregation.")

    # Return both run-level outputs and aggregated summaries so the caller can
    # perform additional analysis without rerunning the workflow.
    return (
        confusions,
        metrics_df,
        avg_confusion,
        avg_importances,
        metrics_summary,
        y_true_runs,
        y_pred_runs,
        mouse_id_runs
    )

In [ ]:
presence_absence_matrix = pd.read_csv('../presence_absence_matrix_final.csv') # read data - PA matrix

# DIET

In [ ]:
def get_condition(data_with_conditions, row_name='Diet'):
    '''
    Extract diet information from the metadata file.
    
    Inputs:
        data_with_conditions : str
            Path to the metadata CSV file containing 'Diet' and 'OmicsEarTag' rows.
        row : int (unused, kept for backward compatibility)
            Placeholder parameter (can be ignored).
    
    Outputs:
        pd.DataFrame - Clean mapping of FAC_ID to Diet
    '''
    # Load metadata
    meta = pd.read_csv(data_with_conditions, index_col=0)
    
    # Extract relevant rows 
    diet_row = meta.loc[row_name]
    fac_row = meta.loc['OmicsEarTag']
    
    # Build mapping
    diet_map = pd.DataFrame({
        'FAC_ID': fac_row.values,
        row_name: diet_row.values
    })
    
    # Drop unwanted rows (header-like entries)
    diet_map = diet_map[~diet_map['FAC_ID'].isin(['Orig_ID', 'IDFemaleAgingColony'])].reset_index(drop=True)
    
    return diet_map

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv") # get diet information from metadata file

In [ ]:
presence_absence_matrix #  check

In [ ]:
def prep_data_for_RF(data, condition):
    """
    Prepare a presence/absence matrix and condition labels for Random Forest.

    Inputs:
        data : pandas.DataFrame
            Metadata table containing sample IDs and the condition of interest.
            Must include a 'FAC_ID' column and the specified `condition` column.

        condition : str
            Name of the column in `data` to use as the target variable
            (for example, diet, cluster, treatment, etc.).

    Returns:
        X : pandas.DataFrame
            Feature matrix containing the presence/absence data only.

        y : pandas.Series
            Target labels for each sample, taken from the selected condition.
    """
    
    # Build a mapping from sample ID to the selected condition label
    condition_map = dict(zip(data['FAC_ID'], data[condition]))
    
    # Create a new row containing the condition label for each sample column
    cluster_row = pd.DataFrame(
        [[condition] + [condition_map.get(col, None) for col in presence_absence_matrix.columns[1:]]],
        columns=presence_absence_matrix.columns
    )
    
    # Add the condition row on top of the presence/absence matrix
    presence_absence_with_conditions = pd.concat(
        [cluster_row, presence_absence_matrix],
        ignore_index=True
    )
    
    # Set the first column ('Row') as the index
    presence_absence_with_conditions = presence_absence_with_conditions.set_index('Row')
    
    # Extract the selected condition row as the target labels
    y = presence_absence_with_conditions.loc[condition].astype(str)
    
    # Remove the condition row so only the feature matrix remains
    X = presence_absence_with_conditions.drop(condition)
    
    # Convert feature values to string format
    X = X.astype(str)
    
    # Print shapes and previews to check that the data was prepared correctly
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("y head:\n", y.head(10))
    print("X head:\n", X.head(10))
    
    return X, y

In [ ]:
print(diet_data) # check
X, y = prep_data_for_RF(diet_data, 'Diet') # prepare data for RF - this will be used for all 3 conditions, just changing the condition name
X = X.sort_index(axis=1) # sort for consistent feature order across runs
X = X.T
X = X.reset_index(drop=True)
model_diet, X_test_diet, y_test_diet, _ = optimize_random_forest_with_PCA_toggle(X, y) 

In [ ]:
X, y = prep_data_for_RF(diet_data, 'Diet') # prepare data for RF - this will be used for all 3 conditions, just changing the condition name
X = X.sort_index(axis=1)
X = X.T
X = X.reset_index(drop=True)
# repeat the optimization and evaluation workflow multiple times to assess stability and aggregate results
confusions_diet, metrics_df_diet, avg_confusion_diet, avg_importances_diet, metrics_summary_diet, y_true_diet, y_pred_diet, mouse_id_runs_diet = repeat_rf_optimization(X, y, n_iter=30, pca_toggle=False, n_runs=50) 

In [ ]:
# Save results (simple, same style as your examples)

metrics_df_diet.to_csv("../PA_matrix_diet_prediction/diet_rf_metrics_per_run.csv", index=False)
metrics_summary_diet.to_csv("../PA_matrix_diet_prediction/diet_rf_metrics_summary.csv", index=False)

pd.DataFrame(avg_confusion_diet).to_csv("../PA_matrix_diet_prediction/diet_rf_avg_confusion_matrix.csv", index=False)

if avg_importances_diet is not None:
    avg_importances_diet.to_csv("../PA_matrix_diet_prediction/diet_rf_avg_feature_importances.csv", index=False)

# save all confusion matrices too (as one file)
np.save("../PA_matrix_diet_prediction/diet_rf_all_confusions.npy", np.array(confusions_diet, dtype=float))

In [ ]:
def make_pred_df(y_true_runs, y_pred_runs, mouse_ids):
    """
    Build a long-format DataFrame containing true labels, predicted labels,
    and mouse IDs for each run.

    Inputs:
        y_true_runs : list
            List of true label arrays or Series, one per run.

        y_pred_runs : list
            List of predicted label arrays or Series, one per run.

        mouse_ids : array-like or list of array-like
            Mouse/sample IDs corresponding to each prediction.
            This can be either:
            - one array used for all runs, if the same IDs appear in the same order
              every time, or
            - a list of arrays, with one array of IDs per run.

    Returns:
        pandas.DataFrame
            A long-format DataFrame with one row per sample prediction and
            the following columns:
            - 'run': run number
            - 'mouse_id': sample ID
            - 'y_true': true label
            - 'y_pred': predicted label
    """
    rows = []

    # Check whether the same mouse IDs should be used for every run
    same_ids_every_run = not isinstance(mouse_ids, (list, tuple))

    # Loop through each run and combine true labels, predicted labels,
    # and mouse IDs into one row per sample
    for run_idx, (yt, yp) in enumerate(zip(y_true_runs, y_pred_runs), start=1):
        yt = np.asarray(yt)
        yp = np.asarray(yp)

        # Use either the shared mouse ID list or the run-specific one
        if same_ids_every_run:
            mids = np.asarray(mouse_ids)
        else:
            mids = np.asarray(mouse_ids[run_idx - 1])

        # Make sure all arrays for this run have matching lengths
        if not (len(yt) == len(yp) == len(mids)):
            raise ValueError(f"Run {run_idx}: length mismatch")

        # Store one record per sample
        for m, t, p in zip(mids, yt, yp):
            rows.append({
                "run": run_idx,
                "mouse_id": m,
                "y_true": t,
                "y_pred": p
            })

    # Convert the collected rows into a DataFrame
    return pd.DataFrame(rows)

In [ ]:
pred_long_diet = make_pred_df(y_true_diet, y_pred_diet, mouse_id_runs_diet)

In [ ]:
pred_long_diet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def hard_mice_error_report(pred_df, threshold=0.5, top_n=20):
    """
    Simple report of mice that are predicted wrongly most often.

    Args:
        pred_df: DataFrame with columns ['mouse_id', 'y_true', 'y_pred']
                 (can also include 'run'; not required)
        threshold: keep mice with error rate > threshold
        top_n: number of hardest mice to show in the plot

    Returns:
        hard_mice: DataFrame with ['mouse_id', 'error_rate', 'n_obs', 'wrong_count']
    """
    df = pred_df.copy()
    df["is_wrong"] = df["y_true"] != df["y_pred"]

    hard_mice = (
        df.groupby("mouse_id", as_index=False)
          .agg(
              error_rate=("is_wrong", "mean"),
              n_obs=("is_wrong", "size"),
              wrong_count=("is_wrong", "sum")
          )
          .sort_values("error_rate", ascending=False)
    )

    # keep only above threshold
    hard_mice = hard_mice[hard_mice["error_rate"] > threshold].reset_index(drop=True)

    print("\nTop wrongly predicted mice:")
    print(hard_mice.head(top_n))

    # plot with same style as analyze_hard_individuals
    if not hard_mice.empty:
        plot_df = hard_mice.head(top_n).sort_values("error_rate", ascending=False)

        plt.figure(figsize=(12, 6))
        sns.barplot(
            data=plot_df,
            x="mouse_id",
            y="error_rate",
            color="salmon",
            edgecolor="black"
        )

        plt.ylabel("Error Rate (0.0 - 1.0)", fontsize=12)
        plt.xlabel("Mouse ID", fontsize=12)
        plt.axhline(y=threshold, color="red", linestyle="--", label=f"Threshold ({threshold})")
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(axis="y", linestyle="--", alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Good news! No mice found with error_rate > {threshold}")

    return hard_mice

In [ ]:
# pred_long_diet must have: mouse_id, y_true, y_pred
hard_mice = hard_mice_error_report(pred_long_diet, threshold=0.40, top_n=20)

In [ ]:
hard_mice

# SEX

In [ ]:
sex_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv", row_name='Sex') # get sex information from metadata file

In [ ]:
print(sex_data)
X, y = prep_data_for_RF(sex_data, 'Sex') # prepare data for RF - this will be used for all 3 conditions, just changing the condition name
X = X.sort_index(axis=1) # sort for consistent feature order across runs
X = X.T
X = X.reset_index(drop=True)
sex_model, X_test_sex, y_test_sex, _ = optimize_random_forest_with_PCA_toggle(X, y, n_iter=20)  # optimize with fewer iterations for speed during testing; increase n_iter for better performance in final runs

In [ ]:
X, y = prep_data_for_RF(sex_data, 'Sex') # prepare data for RF - this will be used for all 3 conditions, just changing the condition name
X = X.sort_index(axis=1)
X = X.T # transpose to have samples as rows and features as columns, which is the expected format for scikit-learn
X = X.reset_index(drop=True) # reset index to ensure it is a simple
sex_model, X_test_sex, y_test_sex, _ = optimize_random_forest_with_PCA_toggle(X, y, pca_toggle=True) # optimize with PCA enabled to see if it improves performance, especially if the feature space is large and may benefit from dimensionality reduction

In [ ]:
repeat_rf_optimization(X, y, n_runs=30, pca_toggle=False, n_iter=50) # repeat the optimization and evaluation workflow multiple times to assess stability and aggregate results

# AGE

In [ ]:
age_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv", row_name='Age') # get age information from metadata file

In [ ]:
presence_absence_matrix = pd.read_csv('../presence_absence_matrix_final.csv')

# Extract model order
model_order = presence_absence_matrix.columns[1:].tolist()

# Align and clean age info
age_map = age_data.set_index('FAC_ID')  # DataFrame with columns FAC_ID and Age
age_vector = pd.to_numeric(age_map.reindex(model_order)['Age'], errors='coerce')

# Define hard cutoffs for age bins
bins = [0, 300, 500, 700, np.inf]
labels = ['<300', '300-500', '500-700', '>700']

# Assign each model to a group
age_groups = pd.cut(age_vector, bins=bins, labels=labels, include_lowest=True)

# Add Age_Group as a new label column only
age_info = pd.DataFrame({
    'FAC_ID': model_order,
    'Age_Group': age_groups.values
})

age_info.to_csv('../age_info.csv')

print(age_info)

In [ ]:
print(age_info)
X, y = prep_data_for_RF(age_info, 'Age_Group') # this are just string so labels
X = X.sort_index(axis=1)
X = X.T
X = X.reset_index(drop=True)
sex_model, X_test_age, y_test_age, _  = optimize_random_forest_with_PCA_toggle(X, y)  # optimize with fewer iterations for speed during testing; increase n_iter for better performance in final runs

In [ ]:
X, y = prep_data_for_RF(age_info, 'Age_Group') # this are just string so labels
X = X.sort_index(axis=1)
X = X.T
X = X.reset_index(drop=True)
sex_model, X_test_age, y_test_age, _  = optimize_random_forest_with_PCA_toggle(X, y, pca_toggle=True) # optimize with PCA enabled to see if it improves performance, especially if the feature space is large and may benefit from dimensionality reduction

In [ ]:
print(presence_absence_matrix.head())

In [ ]:
repeat_rf_optimization(X, y, n_runs=30, pca_toggle=False, n_iter=50) # repeat the optimization and evaluation workflow multiple times to assess stability and aggregate results